# 누적 복습 1. 온라인 광고 클릭 분류

## 난이도
- 간단한 복습 문제

## 데이터셋 설명
- 수업 복습용으로 생성한 가상 온라인 광고 클릭 데이터셋임.
- 실제 고객 데이터가 아니라, 재현 가능한 난수로 만든 학습용 데이터임.
- 모든 feature가 수치형이므로, 지금까지 수업에서 다룬 표 형태 데이터 처리 흐름 그대로 사용할 수 있음.
- target은 `clicked`이며, 사용자가 광고를 클릭했는지 나타냄.
  - `0`: 클릭하지 않음
  - `1`: 클릭함

## 컬럼 설명
- `age`: 사용자 나이
- `daily_site_minutes`: 하루 평균 사이트 이용 시간
- `ad_frequency`: 최근 광고 노출 횟수
- `previous_purchases`: 이전 구매 횟수
- `discount_rate`: 광고에 포함된 할인율
- `page_views`: 최근 페이지 조회 수

## 복습 범위
- feature/target 분리
- 학습/평가 데이터 분리
- 스케일링
- KNN, Logistic Regression, Random Forest 비교
- 분류 모델 평가와 새 데이터 예측


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rng = np.random.default_rng(42)
n_samples = 700

age = rng.integers(18, 65, n_samples)
daily_site_minutes = rng.normal(38, 14, n_samples).clip(3, 100)
ad_frequency = rng.poisson(4.0, n_samples).clip(0, 15)
previous_purchases = rng.poisson(1.4, n_samples).clip(0, 10)
discount_rate = rng.choice([0, 5, 10, 15, 20, 30], n_samples, p=[0.18, 0.20, 0.24, 0.18, 0.14, 0.06])
page_views = rng.normal(8, 4, n_samples).clip(1, 30)

# 광고 클릭 확률을 만들기 위한 가상 규칙.
# 사이트 이용 시간이 길고, 이전 구매가 많고, 할인율이 높으면 클릭 확률이 올라가도록 구성함.
# 광고 노출이 너무 많으면 피로도가 생긴다고 보고 일부 감점 효과를 넣음.
logit = (
        -4.0
        + 0.035 * daily_site_minutes
        + 0.30 * previous_purchases
        + 0.055 * discount_rate
        + 0.08 * page_views
        - 0.08 * ad_frequency
        - 0.012 * (age - 35)
)
click_proba = 1 / (1 + np.exp(-logit))
clicked = rng.binomial(1, click_proba)

ad_df = pd.DataFrame({
    'age': age,
    'daily_site_minutes': daily_site_minutes.round(1),
    'ad_frequency': ad_frequency,
    'previous_purchases': previous_purchases,
    'discount_rate': discount_rate,
    'page_views': page_views.round(1),
    'clicked': clicked
})

X = ad_df.drop('clicked', axis=1)
# print(X)
y = ad_df['clicked']
# print(y)
# print('데이터 크기:', ad_df.shape)
# print('클릭 비율:', y.mean().round(3))
# display(ad_df.head())


## 문제 1. 데이터 분리와 스케일링

광고 클릭 데이터를 학습용과 평가용으로 나누고 스케일링하세요.

### 요구사항
- `train_test_split()` 사용
- `test_size=0.2`, `random_state=42`, `stratify=y` 사용
- `StandardScaler()` 사용
- 학습 데이터에는 `fit_transform()`, 평가 데이터에는 `transform()` 사용
- 분리 결과 shape 출력

### 힌트
- KNN과 Logistic Regression은 feature 값의 크기 차이에 영향을 받을 수 있음.
- `daily_site_minutes`, `discount_rate`, `page_views`처럼 단위가 다른 feature를 비슷한 기준으로 맞추는 과정이 스케일링임.


In [2]:
# 작성: train_test_split()으로 학습/평가 데이터를 분리하세요.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)
# 작성: StandardScaler()로 학습/평가 데이터를 스케일링하세요.
# 스케일 생성
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 작성: 분리 결과 shape를 출력하세요.
print("X_train : ", X_train.shape)
print("X_test : ", X_test.shape)
print("X_train_scaled : ", X_train_scaled.shape)
print("X_test_scaled : ", X_test_scaled.shape)

X_train :  (560, 6)
X_test :  (140, 6)
X_train_scaled :  (560, 6)
X_test_scaled :  (140, 6)


## 문제 2. 분류 모델 3개 성능 비교

KNN, Logistic Regression, Random Forest를 학습하고 평가 데이터 accuracy를 비교하세요.

### 요구사항
- `KNeighborsClassifier(n_neighbors=5)` 사용
- `LogisticRegression(max_iter=2000, random_state=42)` 사용
- `RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)` 사용
- 결과 컬럼: `model`, `train_accuracy`, `test_accuracy`
- 결과를 DataFrame으로 출력

### 힌트
- KNN은 가까운 이웃을 보고 예측함.
- Logistic Regression은 선형식으로 class 확률을 계산함.
- Random Forest는 여러 결정트리의 투표로 최종 class를 예측함.


In [3]:
# 작성: KNN, Logistic Regression, Random Forest 모델을 딕셔너리로 준비하세요.
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
}

results = []

# 작성: 반복문으로 모델을 학습하고 train/test accuracy를 results에 저장하세요.
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    train_accuracy = accuracy_score(y_train, model.predict(X_train_scaled))
    test_accuracy = accuracy_score(y_test, y_pred)

    results.append({
        'model': name,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
    })

# 작성: 결과를 DataFrame으로 출력하세요.
results_df = pd.DataFrame(results)
display(results_df)


,model,train_accuracy,test_accuracy
0,KNN,0.812500,0.728571
1,Logistic Regression,0.766071,0.778571
2,Random Forest,0.814286,0.771429


## 문제 3. 최종 모델 선택과 새 데이터 예측

평가 accuracy가 가장 높은 모델을 골라 상세 결과를 확인하고, 새 사용자 2명의 광고 클릭 여부를 예측하세요.

### 요구사항
- 문제 2 결과에서 가장 좋은 모델 선택
- `classification_report()` 출력
- 새 사용자 2명 데이터를 DataFrame으로 생성
- 새 데이터에도 기존 scaler의 `transform()` 적용
- 예측 class와 클릭 확률을 출력

### 힌트
- `predict()`는 최종 class를 반환함.
- `predict_proba()`는 class별 확률을 반환함.
- 새 데이터도 학습 데이터와 같은 컬럼 순서로 만들어야 함.


In [13]:
# 작성: result_df에서 평가 accuracy가 가장 높은 모델 이름을 찾으세요.
best_model_name = results_df.loc[results_df['test_accuracy'].idxmax(), 'model']
print(best_model_name)

y_pred = models[best_model_name].predict(X_test_scaled)
print(y_pred)
# 작성: 선택한 모델로 평가 데이터를 예측하고 classification_report()를 출력하세요.
# print(ad_df)
# print(ad_df.target_names)
# print(classification_report(y_test, y_pred, target_names=['not_clicked', 'clicked']))
print(classification_report(y_test, y_pred, target_names=['not_clicked', 'clicked']))

# 작성: 새 사용자 2명의 데이터를 만들고 클릭 여부와 클릭 확률을 예측하세요. ?? 이건 확인해보기
new_users = pd.DataFrame({
    'age': [28, 52],
    'daily_site_minutes': [55, 18],
    'ad_frequency': [3, 10],
    'previous_purchases': [3, 0],
    'discount_rate': [20, 5],
    'page_views': [14, 4]
})

# 추가된 새로운 유저데이터를
# 학습 때 사용한 scaler로 새 사용자 데이터도 변환
new_users_scaled = scaler.transform(new_users)

# 가장 좋은 모데을 가져와서
best_model = models[best_model_name]
# 추가된 학습 데이터로 최고의 모델을 적용하여 예측값을 확인함
new_pred = best_model.predict(new_users_scaled)
print(new_pred)
# 클릭학 예측 확률을 구함
new_pred_proba = best_model.predict_proba(new_users_scaled)
clicked_proba = new_pred_proba[:, 1]

# 데이터 프레임에 새로운 유저의 확률을 추가
new_data = new_users.copy()
new_data['클릭여부'] = new_pred
new_data['클릭여부에 대한 확률'] = clicked_proba

new_data_df = pd.DataFrame(new_data)
display(new_data_df)




Logistic Regression
[0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
              precision    recall  f1-score   support

 not_clicked       0.78      0.98      0.87       107
     clicked       0.67      0.12      0.21        33

    accuracy                           0.78       140
   macro avg       0.73      0.55      0.54       140
weighted avg       0.76      0.78      0.71       140

[1 0]


,age,daily_site_minutes,ad_frequency,previous_purchases,discount_rate,page_views,클릭여부,클릭여부에 대한 확률
0,28,55,3,3,20,14,1,0.682857
1,52,18,10,0,5,4,0,0.033925


In [17]:
new_customers = pd.DataFrame({
  'age': [24, 45, 60],
  'daily_site_minutes': [62, 35, 12],
  'ad_frequency': [2, 6, 11],
  'previous_purchases': [4, 1, 0],
  'discount_rate': [30, 10, 5],
  'page_views': [18, 7, 3]
})

# 문제 1. 새 사용자 데이터에 스케일러를 적용하세요.
cus_scaled = scaler.transform(new_customers)

# 문제 2. 가장 좋은 모델을 변수로 꺼내세요.
best_model = models[best_model_name]

# 문제 3. 새 사용자 3명의 예측 class를 구하세요.
# 새롭게 추가된 모델의 클릭여부의 대한 예상값을 구한다
new_cus_pred = best_model.predict(cus_scaled)

# 문제 4. 새 사용자 3명의 class별 예측 확률을 구하세요.
# 새로운 유저의 값을 보고 예상치의 확률을 모두 구함
new_cus_proba = best_model.predict_proba(cus_scaled)

# 0열 : 클릭안할확률 , 1열이 : 클릭할 확률
# 모든 행을 가져오되 클릭할 확률인 1번 열을 모두 가져온다
click_probas = new_cus_proba[:, 1]

# 새로운 유저의 데이터를 복사 후
new_cus_results = new_customers.copy()
# 컬럼 추가
# 클릭 예상 여부 컬럼 추가 및 값 입력
new_cus_results['클릭'] = new_cus_pred
# 클릭 할 확률과 안할 확률의 확률을 입력
new_cus_results['클릭여부 확률'] = click_probas
# 데이터 프레임 출력
display(new_cus_results)

,age,daily_site_minutes,ad_frequency,previous_purchases,discount_rate,page_views,클릭,클릭여부 확률
0,24,62,2,4,30,18,1,0.882030
1,45,35,6,1,10,7,0,0.144634
2,60,12,11,0,5,3,0,0.020628


In [21]:
trial_users = pd.DataFrame({
      'age': [31, 39, 57, 22],
      'daily_site_minutes': [48, 20, 75, 9],
      'ad_frequency': [4, 12, 3, 8],
      'previous_purchases': [2, 0, 5, 0],
      'discount_rate': [15, 0, 30, 10],
      'page_views': [10, 3, 22, 2]
  })

# 문제 1. 새 사용자 데이터에 스케일러를 적용하세요.
trial_scaled = scaler.transform(trial_users)
# 문제 2. 가장 좋은 모델을 변수로 꺼내세요.
model_name = results_df.loc[results_df['test_accuracy'].idxmax(), 'model']
best = models[model_name]
print(best)
# 문제 3. 새 사용자 4명의 예측 class를 구하세요.
# 새로운 유저의 미리지정한 스케일로 변환 후 값을 제일 적합한 모델 넣는다
trial_pred = best.predict(trial_scaled)

# 문제 4. 새 사용자 4명의 class별 예측 확률을 구하세요.
trial_pred_proba = best.predict_proba(trial_scaled)


# • 문제 5. 클릭할 확률만 따로 꺼내세요.
trial_probas = trial_pred_proba[:, 1]

# 문제 6. 원본 새 사용자 데이터에 예측 결과를 추가하세요.
trial_results = trial_users.copy()
trial_results["클릭여부"] = trial_pred
trial_results["클릭여부의 확률"] = trial_probas

trial_results_df = pd.DataFrame(trial_results)
display(trial_results_df)

print(trial_results)

LogisticRegression(max_iter=2000, random_state=42)
   age  daily_site_minutes  ad_frequency  previous_purchases  discount_rate  \
0   31                  48             4                   2             15   
1   39                  20            12                   0              0   
2   57                  75             3                   5             30   
3   22                   9             8                   0             10   

   page_views  클릭여부  클릭여부의 확률  
0          10     0  0.425448  
1           3     0  0.033303  
2          22     1  0.896732  
3           2     0  0.055944  
